# 多进程

Python 使用 `multiprocessing` 模块创建多进程。

## 使用 Process 类创建进程

In [ ]:
from multiprocessing import Process
import os

# 子进程要执行的函数
def run_proc(name):
    print('Run child process %s (%s)...' % (name, os.getpid()))

if __name__ == '__main__':
    print('Parent process %s.' % os.getpid())
    
    # 创建进程
    p = Process(target=run_proc, args=('test',))
    print('Child process will start.')
    
    # 启动进程
    p.start()
    
    # 等待子进程结束
    p.join()
    print('Child process ended.')

## 使用进程池 Pool

In [ ]:
from multiprocessing import Pool
import os, time, random

def long_time_task(name):
    print('Run task %s (%s)...' % (name, os.getpid()))
    start = time.time()
    time.sleep(random.random() * 3)
    end = time.time()
    print('Task %s runs %0.2f seconds.' % (name, (end - start)))

if __name__ == '__main__':
    print('Parent process %s.' % os.getpid())
    
    # 创建进程池，最大 4 个进程
    p = Pool(4)
    
    # 提交 5 个任务
    for i in range(5):
        p.apply_async(long_time_task, args=(i,))
    
    print('Waiting for all subprocesses done...')
    p.close()
    p.join()
    print('All subprocesses done.')

## 进程间通信 - Queue

In [ ]:
from multiprocessing import Process, Queue
import os, time, random

# 写进程
def _w(q):
    print('Process to write: %s' % os.getpid())
    for value in ['A', 'B', 'C']:
        print('Put %s to queue...' % value)
        q.put(value)
        time.sleep(random.random())

# 读进程
def _r(q):
    print('Process to read: %s' % os.getpid())
    while True:
        value = q.get(True)
        print('Get %s from queue.' % value)

if __name__ == '__main__':
    q = Queue()
    pw = Process(target=_w, args=(q,))
    pr = Process(target=_r, args=(q,))
    
    pw.start()
    pr.start()
    
    pw.join()
    pr.terminate()
    print('Done.')

## 小结

- `Process` 类用于创建单个进程
- `Pool` 类用于创建进程池，批量处理任务
- `Queue` 用于进程间通信
- 在 Windows 上，多进程代码必须放在 `if __name__ == '__main__':` 块中